# **응급상황 자동 인식 및 응급실 연계 서비스**
# **단계3 : 응급상황 연계(추천)**

## **0.미션**

단계 3에서는, 응급상황의 음성을 인식해서 텍스트로 변환하고, 변환된 텍스트를 다시 요약 및 핵심키워드 도출 작업을 수행합니다.  
이를 위해 사전학습된 모델을 API로 연결하여 활용합니다.

### 미션4 : 응급실 추천
* 응급실 위치와 응급전화 발신자 위치 기반 추천
* 두 좌표간 직선거리(Haversine)
    * 1) 500여 곳 응급실에 대해서, 거리 기반 가까운 응급실 찾기
    * 2) 좌표 구간을 설정하여 대상 응급실 범위를 좁힌 후, 거리 기반 가까운 응급실 찾기


## **1.환경설정**

### (1) 경로 설정

구글 드라이브 연결

#### 1) 구글 드라이브 폴더 생성
* 새 폴더(project6_2)를 생성하고
* 제공 받은 파일을 업로드

#### 2) 구글 드라이브 연결

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
path = '/content/drive/MyDrive/# KT aivle school/# 수업 코드/#17 Mini Project 6th-2/'

### (2) 라이브러리

#### 1) 필요한 라이브러리 설치

* requirements.txt 파일의 [경로 복사]를 한 후,
* 아래 경로에 붙여 넣기

In [4]:
# 경로 : /content/drive/MyDrive/project6_2/requirements.txt
# 경로가 다른 경우 아래 코드의 경로 부분을 수정하세요.

!pip install -r '/content/drive/MyDrive/# KT aivle school/# 수업 코드/#17 Mini Project 6th-2/requirements.txt'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 16.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


#### 2) 라이브러리 로딩

In [43]:
#필요한 라이브러리 설치 및 불러우기
import os
import pandas as pd
import numpy as np

from haversine import haversine
import requests
import json

# 더 필요한 라이브러리 추가 -------------
from tqdm import tqdm



### (3) 데이터 로딩
* 단계1에서 수집한 응급실 정보를 불러와서 데이터프레임으로 저장합니다.

In [51]:
df_emerg = pd.read_csv(path + '응급실 정보.csv', index_col=0)
df_emerg.reset_index(drop=True, inplace=True)
df_emerg.head()

,주소,응급의료기관 종류,전화번호 1,전화번호 3,위도,경도
0,울산광역시 남구 남산로354번길 26 (신정동),응급실운영신고기관,052-220-3300,052-220-3334,35.548238,129.307011
1,부산광역시 기장군 기장읍 대청로72번길 6,지역응급의료기관,051-723-0171,051-723-2119,35.236029,129.216492
2,"인천광역시 서구 칠천왕로33번길 17 (석남동, 신석로 70(석남1동, 성민병원))",지역응급의료기관,032-726-1000,032-726-1190,37.508994,126.669479
3,"경기도 용인시 처인구 백옥대로1082번길 18, 다보스종합병원 (김량장동)",지역응급의료센터,031-8021-2114,031-8021-2130,37.234641,127.210499
4,경기도 용인시 처인구 고림로 81 (고림동),지역응급의료기관,031-337-0114,031-336-0119,37.240316,127.214491


## **2. 응급실 추천**


### (1) 직선거리 계산
- haversine formula
    * Haversine은 두 지점 간의 거리를 구할 때 사용하는 수학 공식으로, 지구의 구형 구조를 고려하여 위도와 경도를 기반으로 직선 거리를 계산한다.
- 세부사항
    * 하버사인 함수를 활용


#### 1) 하버사인 함수 사용 연습
* 임의의 두 좌표간 거리 계산
    * 응급실 데이터프레임을 열어서
    * 응급실 두 곳의 좌표를 확인하고
    * 두 지점의 거리를 계산해 봅시다
* 사용법 : haversine((위도1, 경도1), (위도2, 경도2), unit='km')


In [24]:
point1 = df_emerg.loc[0, ['위도', '경도']]
point2 = df_emerg.loc[1, ['위도', '경도']]
haversine((point1['위도'], point1['경도']), (point2['위도'], point2['경도']), unit='km')

7.379665041345399

#### 2) 가장 가까운 응급실 3곳 추천하기1
* 세부사항
    * 입력된 좌표와 전체 응급실과 거리를 계산한 후
    * 가장 가까운 거리의 응급실 3 곳을 반환합니다.
* 이를 함수로 생성하고 테스트 해 봅시다.

In [52]:
df_loc = pd.read_excel(path + 'audio_location.xlsx', index_col=0)
start = df_loc[['위도', '경도']].iloc[0]

In [75]:
def recommend_hospital1(start, df_emerg):
  for i in range(len(df_emerg)):
    end = df_emerg[['위도', '경도']].iloc[i]
    df_emerg.loc[i, '거리'] = haversine((start['위도'], start['경도']), (end['위도'], end['경도']), unit='km')
  df_emerg = df_emerg.sort_values(by='거리')
  return df_emerg.head(3)

df_recommend1 = recommend_hospital1(start, df_emerg)
df_recommend1.head()

,주소,응급의료기관 종류,전화번호 1,전화번호 3,위도,경도,거리
156,"경기도 성남시 분당구 구미로173번길 82 (구미동, 분당서울대학교병원)",권역응급의료센터,031-787-2114,031-787-3119,37.352026,127.124484,1.111162
116,경기도 성남시 분당구 서현로180번길 20 (서현동),지역응급의료센터,031-779-0114,031-779-0119,37.387871,127.121328,3.299964
76,경기도 성남시 분당구 새마을로177번길 81 (율동),지역응급의료기관,031-725-6075,031-725-6119,37.391867,127.148586,4.738899


#### 3) 가장 가까운 응급실 3곳 추천하기2
* 문제점 : 입력 받은 좌표와 응급실 전체와의 거리를 모두 계산하는 것은 비효율 적입니다.
* 해결 방안 : 그래서 입력 받은 좌표를 기준으로 일정 범위 내에 해당되는 응급실에 대해서 거리를 계산하고 추천하도록 기존 함수를 수정 합니다.
* hint :
    * 입력 받은 위도, 경도 값에 ± α 하여 일정 범위 구간을 정하고
    * 응급실 정보에서 해당 구간을 먼저 조회한 후
    * 거리 계산

In [76]:
df_loc = pd.read_excel(path + 'audio_location.xlsx', index_col=0)
start = df_loc[['위도', '경도']].iloc[0]

In [88]:
def recommend_hospital2(start, df_emerg, a_lat, a_lng):
  # 필요한거 준비
  start_lat, start_lng = start[['위도', '경도']]
  df_emerg = df_emerg[(start_lat -  a_lat < df_emerg['위도']) &
   (df_emerg['위도'] < start_lat +  a_lat) &
   (start_lng -  a_lng < df_emerg['경도']) &
    (df_emerg['경도'] < start_lng +  a_lng)]
  df_emerg.reset_index(drop=True, inplace=True)

  # 거리 계산
  temp = []
  for i in range(3):
    dest_lat, dest_lng = df_emerg.loc[i, ['위도', '경도']]
    df_emerg.loc[i, '거리'] = haversine((start_lat, start_lng), (dest_lat, dest_lng), unit='km')
  df_emerg = df_emerg.sort_values(by='거리')

  return df_emerg.head(3)

df_recommend2 = recommend_hospital2(start, df_emerg, 0.5, 0.8)
df_recommend2

,주소,응급의료기관 종류,전화번호 1,전화번호 3,위도,경도,거리
60,"경기도 성남시 분당구 구미로173번길 82 (구미동, 분당서울대학교병원)",권역응급의료센터,031-787-2114,031-787-3119,37.352026,127.124484,1.111162
46,경기도 성남시 분당구 서현로180번길 20 (서현동),지역응급의료센터,031-779-0114,031-779-0119,37.387871,127.121328,3.299964
32,경기도 성남시 분당구 새마을로177번길 81 (율동),지역응급의료기관,031-725-6075,031-725-6119,37.391867,127.148586,4.738899


### (2) [조 과제]고도화 : naver 지도 api 사용

* 이 부분은 조별 과제로 수행하게 됩니다.(개인과제 아님!)

* 세부사항
    * 두 지점간, 최단 도로거리, 소요 시간을 계산하는 함수를 생성하시오.
    * 함수 내용
        * 입력 : 두 지점의 위도, 경도, 네이버클라우드id, 암호키
        * 출력 : 도로거리(km)
    
    * 네이버 Maps API 활용
        * 사용할 API : Direction 5
        * 가이드 : https://guide.ncloud-docs.com/docs/ko/maps-direction5-api
        * 가이드를 활용해서 url, header, params를 구성합니다.
        * params의 옵션은 'trafast' (실시간 빠른 길 옵션)을 선택하시오.

#### 1) maps 클라이언트ID, 키 로딩

In [66]:
c_id = 'nf30b2d7do'
c_key = 'Ao1BZYROhTsZ72MeOCJgtAeoAZbqbzGFy2UXMo5h'

#### 2) 함수 생성

In [59]:
def get_dist(start_lat, start_lng, dest_lat, dest_lng, c_id, c_key):
    url = "https://naveropenapi.apigw.ntruss.com/map-direction/v1/driving"
    headers = {
        "X-NCP-APIGW-API-KEY-ID": c_id,
        "X-NCP-APIGW-API-KEY": c_key,
    }
    params = {
        "start": f"{start_lng},{start_lat}",  # 출발지 (경도, 위도)
        "goal": f"{dest_lng},{dest_lat}",    # 목적지 (경도, 위도)
        "option": "trafast"  # 실시간 빠른 길 옵션
    }

    # 요청하고, 답변 받아오기
    response = requests.get(url, headers=headers, params=params)

    if response.status_code == 200:
        response = response.json()
    else:
        raise Exception(f"API 요청 실패: {response.status_code} - {response.text}")

    dist = response['route']['trafast'][0]['summary']['distance']  # m(미터)
    dist = dist / 1000  # km로 변환
    return dist

* 테스트

In [60]:
start_lat, start_lng = df_loc[['위도', '경도']].iloc[0]
dest_lat, dest_lng = df_emerg.loc[0, ['위도', '경도']]
dist = get_dist(start_lat, start_lng, dest_lat, dest_lng, c_id, c_key)
print(dist)

355.207


#### 3) 응급실 추천
* recommend_hospital2 함수를 참조해서 recommend_hospital3 만들기
    * 거리 계산 부분을 get_dist 함수로 대체
    * 입력 부분 수정

In [68]:
dest_lat, dest_lng = df_emerg.loc[0, ['위도', '경도']]
print(dest_lat, dest_lng)

35.54823820112527 129.30701143429678


In [89]:
def recommend_hospital3(start_lat, start_lng, df_emerg, a_lat, a_lng, c_id, c_key):
  # 필요한거 준비
  df_emerg = df_emerg[(start_lat -  a_lat < df_emerg['위도']) &
   (df_emerg['위도'] < start_lat +  a_lat) &
   (start_lng -  a_lng < df_emerg['경도']) &
    (df_emerg['경도'] < start_lng +  a_lng)]
  df_emerg.reset_index(drop=True, inplace=True)

  # 거리 계산
  for i in range(3):
    dest_lat, dest_lng = df_emerg.loc[i, ['위도', '경도']]
    df_emerg.loc[i, '거리'] = get_dist(start_lat, start_lng, dest_lat, dest_lng, c_id, c_key)
  df_emerg = df_emerg.sort_values(by='거리')

  return df_emerg.head(3)
df_recommend3 = recommend_hospital3(start_lat, start_lng, df_emerg, 0.2, 0.4, c_id, c_key)

In [90]:
df_recommend3

,주소,응급의료기관 종류,전화번호 1,전화번호 3,위도,경도,거리
39,"경기도 성남시 분당구 구미로173번길 82 (구미동, 분당서울대학교병원)",권역응급의료센터,031-787-2114,031-787-3119,37.352026,127.124484,1.111162
28,경기도 성남시 분당구 서현로180번길 20 (서현동),지역응급의료센터,031-779-0114,031-779-0119,37.387871,127.121328,3.299964
21,경기도 성남시 분당구 새마을로177번길 81 (율동),지역응급의료기관,031-725-6075,031-725-6119,37.391867,127.148586,4.738899


## **Mission Complete!**

수고 많았습니다!